# 08 V3 — Simple spherical digital phantom

This version supplies **known density, B0 and relaxation** directly to the Generation 08 MRzero model. It skips the measured digital-twin preparation, 32-echo fitting and measured GRE target.

Generation 08 acquires one 2D slice. We use a **voxelised axial cross-section of a sphere**, with one finite full-thickness voxel per pixel and the same three-sinc dephasing model. The central slice is a disk. The curved sphere boundary inside an edge voxel is approximated by that uniform box; the full 3D sphere and an RF slice profile are not simulated.

The only input files are the EPI **`.seq`** and the **3D ACDC calibration**. The calibration supplies the existing 32 shim-coil fields; the object itself is entirely synthetic. An **air bubble below the slice** now supplies a known B0 perturbation, including its through-slice gradient. Set `BUBBLE_RADIUS_MM = 0.` to recover the unperturbed test.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from contextlib import redirect_stdout
import json, sys, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.ndimage import binary_erosion
from IPython.display import display, Markdown, clear_output

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/"data/ACDC_3T.npz").is_file()
                     and (p/"MRzero_generations_08_09").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the ACDC project.")
HELPER_DIR = PROJECT_ROOT/"MRzero_generations_08_09"
sys.path.insert(0, str(HELPER_DIR))
from read_acdc_sequence import read_sequence
from sphere_phantom_08 import make_sphere_inputs
from voxel_3d import FiniteVoxel3DACDCModel
from experiment_optimization import optimize_mrzero_image
import continuous_reconstruction
continuous_reconstruction.FORCE_DIRECT_DFT = False
continuous_reconstruction.MATRIX_DFT_ENABLED = False
torch.set_num_threads(2)

def table(values):
    display(Markdown("| Parameter | Value |\n| --- | --- |\n" +
                     "\n".join(f"| {key} | {value} |" for key, value in values.items())))

def image_ax(ax, values, title, **options):
    artist = ax.pcolormesh(phantom["X_m"]*1000, phantom["Y_m"]*1000,
                          values, shading="auto", rasterized=True, **options)
    ax.set(title=title, xlabel="X [mm]", ylabel="Y [mm]", aspect="equal")
    return artist

def nrmse_percent(image, reference):
    return float(100*np.linalg.norm(np.abs(image)-np.abs(reference))/np.linalg.norm(reference))

## 1. Parameters

The initial run uses a **4 mm radius air bubble**, centred at **(15, -10, -18) mm** relative to the phantom centre. It lies entirely below the 5 mm slab, so there is no air hole in the imaged density. The assumed air–water susceptibility contrast is +9.4 ppm (SI), at 3 T, with the main field along Z. These are configurable simulation parameters, not measured values. T2* remains infinite to isolate the imposed B0 effect.

Outside a spherical bubble, the weak-susceptibility field is

$$\Delta f(\mathbf r)=\frac{\gamma_{\rm Hz/T}B_{\rm main}\Delta\chi a^3}{3}
\frac{3(\hat{\mathbf b}\cdot\mathbf R)^2-|\mathbf R|^2}{|\mathbf R|^5},
\qquad \mathbf R=\mathbf r-\mathbf r_{\rm bubble}.$$

The helper evaluates this field and its analytic X/Y/Z gradient. The existing three-sinc model integrates the **local linear approximation**, not the full curved field. We report its error against the exact field at all voxel corners. See [the spherical-field formula, Eq. 3](https://pmc.ncbi.nlm.nih.gov/articles/PMC3465725/). The large phantom sphere's own susceptibility boundary is not simulated.

`B0_OFFSET_HZ` and `B0_GRADIENT_HZ_PER_M` add an optional linear background; gradient entries are X/Y/Z in Hz/m.

`SLICE_OFFSET_MM` selects a plane relative to the sphere centre. `COIL_CENTER_MM` is its virtual placement in the calibration volume. Synthetic X/Y align directly with calibration X/Y, with no rotation from the measured experiment. Complete calibration coverage is required; uncovered sphere pixels are not silently cropped.

The existing 08 reconstruction follows the Pulseq ADC coordinates. GPU selection is automatic, with CPU allowed. Optimization is optional and initially disabled.

In [ ]:
SEQUENCE_PATH = PROJECT_ROOT/"acquisition_029_(06)/sequences/gre_epi_128x128_fov200mm_nav3_cimax_snr_t2star40ms_line_triggers_v6.seq"
CALIBRATION_PATH = PROJECT_ROOT/"data/ACDC_3T.npz"

SPHERE_RADIUS_MM = 60.
SLICE_OFFSET_MM = 0.
SLICE_THICKNESS_MM = 5.
PROTON_DENSITY = 1.
T2STAR_MS = np.inf                 # Try 40. for uniform exponential decay.
B0_OFFSET_HZ = 0.
B0_GRADIENT_HZ_PER_M = (0., 0., 0.) # Try (300., -200., 0.) for a field test.
COIL_CENTER_MM = (0., 0., 30.)
BUBBLE_RADIUS_MM = 4.             # Set 0. to disable the bubble.
BUBBLE_CENTER_MM = (15., -10., -18.) # Phantom coordinates; below the slice.
BUBBLE_DELTA_CHI_PPM = 9.4         # Assumed chi_air - chi_water, SI ppm.
MAIN_FIELD_T = 3.
MAIN_FIELD_DIRECTION = (0., 0., 1.) # X/Y/Z direction in phantom coordinates.

GPU_PROFILE = "auto"              # "cpu" also works.
GPU_PRECISION = "float64"          # Native MRzero remains float32.
MAX_CURRENT_A = 3.
DISABLED_CHANNELS = [9]            # Physical outputs, numbered 1 to 32.
TOTAL_CURRENT_LIMIT_A = 30.
FIELD_LIMIT_HZ = 1000.
RUN_OPTIMIZATION = False
MAX_ITERATIONS = 20

OUTPUT_DIR = HELPER_DIR/"results"/datetime.now(ZoneInfo("Europe/Paris")).strftime("mrzero08_V3_sphere_%Y%m%d_%H%M%S_%f")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results:", OUTPUT_DIR)

## 2. Create the known object

The sequence reader provides the imaging ADC coordinates, RF-relative times and original EPI line ordering. The sphere helper supplies the object maps directly. It samples the calibrated ACDC fields at each voxel centre, the six edge endpoints and all eight corners.

No scan data or echo fit is used. Density is defined **at RF time** (`amplitude_reference_time_s = 0`), and there is no fitted unresolved-coherence factor.

In [ ]:
acquisition, timing = read_sequence(SEQUENCE_PATH)
if acquisition["sequence_kind"] != "epi":
    raise ValueError("Choose a single-shot EPI sequence for this notebook.")
experiment, calibration, phantom = make_sphere_inputs(
    acquisition, CALIBRATION_PATH,
    radius_m=SPHERE_RADIUS_MM*1e-3,
    slice_offset_m=SLICE_OFFSET_MM*1e-3,
    slice_thickness_m=SLICE_THICKNESS_MM*1e-3,
    proton_density=PROTON_DENSITY, t2star_s=T2STAR_MS*1e-3,
    b0_offset_Hz=B0_OFFSET_HZ, b0_gradient_Hz_per_m=B0_GRADIENT_HZ_PER_M,
    coil_center_m=np.asarray(COIL_CENTER_MM)*1e-3,
    bubble_radius_m=BUBBLE_RADIUS_MM*1e-3,
    bubble_center_m=np.asarray(BUBBLE_CENTER_MM)*1e-3,
    bubble_delta_chi_ppm=BUBBLE_DELTA_CHI_PPM,
    main_field_T=MAIN_FIELD_T, main_field_direction=MAIN_FIELD_DIRECTION)
table({"Image matrix": acquisition["kx_mInv"].shape,
       "FOV [mm]": np.asarray(acquisition["FOV_m"])*1000,
       "Sphere radius [mm]": SPHERE_RADIUS_MM,
       "Slice offset / thickness [mm]": (SLICE_OFFSET_MM, SLICE_THICKNESS_MM),
       "Retained voxels / excluded voxels": (phantom["retained_voxels"], phantom["excluded_calibration_voxels"]),
       "ACDC channels": calibration["acdc_maps"].shape[-1],
       "Virtual sphere centre in coil volume [mm]": COIL_CENTER_MM,
       "ACDC corner linearisation error [Hz/A]": phantom["corner_linearization_max_error_Hz_per_A"],
       "Bubble radius / centre [mm]": (BUBBLE_RADIUS_MM, BUBBLE_CENTER_MM),
       "B0 range in slice [Hz]": (np.nanmin(phantom["B0_Hz"]), np.nanmax(phantom["B0_Hz"])),
       "Bubble corner linearisation error [Hz]": phantom["bubble"]["corner_linearization_max_error_Hz"]})

In [ ]:
fig = plt.figure(figsize=(16, 3.8), layout="constrained")
ax = fig.add_subplot(141, projection="3d")
u, v = np.meshgrid(np.linspace(0, 2*np.pi, 36), np.linspace(0, np.pi, 19))
r = SPHERE_RADIUS_MM
ax.plot_wireframe(r*np.cos(u)*np.sin(v), r*np.sin(u)*np.sin(v), r*np.cos(v),
                  color="steelblue", linewidth=.45, rstride=2, cstride=2)
xx, yy = np.meshgrid([-r, r], [-r, r])
for z in (SLICE_OFFSET_MM-SLICE_THICKNESS_MM/2, SLICE_OFFSET_MM+SLICE_THICKNESS_MM/2):
    ax.plot_surface(xx, yy, np.full(xx.shape, z), color="orange", alpha=.22)
if BUBBLE_RADIUS_MM > 0:
    bx, by, bz = BUBBLE_CENTER_MM
    a = BUBBLE_RADIUS_MM
    ax.plot_surface(bx+a*np.cos(u)*np.sin(v), by+a*np.sin(u)*np.sin(v), bz+a*np.cos(v),
                    color="crimson", alpha=.9)
    ax.text(bx, by, bz-10, "Air bubble", color="crimson", fontsize=8)
ax.set(title="Sphere, slab and bubble", xlabel="X [mm]", ylabel="Y [mm]", zlabel="Z [mm]")
ax.set_box_aspect((1, 1, 1))
ax = fig.add_subplot(142)
im = image_ax(ax, phantom["PD"], "Known slice density", cmap="gray", vmin=0, vmax=PROTON_DENSITY)
fig.colorbar(im, ax=ax, label="PD [a.u.]", shrink=.8)
ax = fig.add_subplot(143)
limit = max(1., float(np.nanmax(abs(phantom["B0_Hz"]))))
im = image_ax(ax, phantom["B0_Hz"], "Total B0 offset", cmap="RdBu_r", vmin=-limit, vmax=limit)
fig.colorbar(im, ax=ax, label="Hz", shrink=.8)
ax = fig.add_subplot(144)
gz = phantom["total_B0_gradient_Hz_per_m"][..., 2]/1000
limit = max(.001, float(np.nanmax(abs(gz))))
im = image_ax(ax, gz, "Through-slice gradient", cmap="RdBu_r", vmin=-limit, vmax=limit)
fig.colorbar(im, ax=ax, label="Hz/mm", shrink=.8)
fig.savefig(OUTPUT_DIR/"sphere_inputs.png", dpi=300)
plt.show(); plt.close(fig)

## 3. Run the Generation 08 model

The model uses the same native MRzero centre histories, analytic three-axis box integral and hardware response as 08. The sequence supplies encoding and timing; excitation is the existing ideal 90° model. Spins are stationary, transmit and receive sensitivity are uniform, and diffusion is zero.

The model also produces its **zero-B0, zero-current reference** with the same density, finite voxel boxes and T2*. This numerical image is the target; the binary density map above is separate ground truth. Finite voxel encoding and finite T2* can alter the reconstructed edge even when B0 is zero.

In [ ]:
channel_limits = np.full(calibration["acdc_maps"].shape[-1], MAX_CURRENT_A)
for channel in DISABLED_CHANNELS:
    if not 1 <= channel <= len(channel_limits):
        raise ValueError("A disabled channel is outside the physical output range.")
    channel_limits[channel-1] = 0.

started = time.perf_counter()
with (OUTPUT_DIR/"mrzero_simulation.log").open("w") as log, redirect_stdout(log):
    model = FiniteVoxel3DACDCModel(
        experiment, calibration, max_current_A=channel_limits,
        profile=GPU_PROFILE, precision=GPU_PRECISION, require_cuda=False,
        memory_fraction=.4, sample_chunk=128,
        hardware={"min_update_interval_s": 8e-6, "dac_resolution_A": .0006})
    zero_currents = np.zeros((model.nc, model.nstates))
    baseline_image, baseline_kdata = model.forward(zero_currents)
model.runtime_info.update(object_amplitude="Known synthetic PD at RF",
                          amplitude_reference_time_s=0.,
                          amplitude_at_RF="PD supplied directly; no echo extrapolation",
                          receiver_model="One uniform receive channel")
reference_image = model.ideal_image
elapsed_s = time.perf_counter()-started
clear_output(wait=True)
print(f"MRzero setup and zero-current simulation completed in {elapsed_s:.2f} s.")
print("Python simulation log:", OUTPUT_DIR/"mrzero_simulation.log")

## 4. First-simulation parameters and checks

These values describe the actual model just constructed. B0 includes the known air-bubble field and optional linear background. No frequency width or through-slice gradient is inferred from measured echoes. B0 and ACDC fields are linearised inside each voxel; their corner mismatches quantify the neglected curvature. The quadrature check below verifies the three-sinc integral of that linear model, not an exact integral of the curved bubble field.

Image NRMSE uses the complete FOV and the numerical zero-B0 reference, with **no fitted gain**. The interior CV is a descriptive variation metric on the sphere mask eroded by two pixels; finite support and voxel windows still affect it.

In [ ]:
interior = binary_erosion(phantom["support"], iterations=2)
if not interior.any():
    interior = phantom["support"].copy()

def interior_cv_percent(image):
    values = np.abs(image)[interior]
    return float(100*values.std()/values.mean())

metrics = {"baseline_NRMSE_percent": nrmse_percent(baseline_image, reference_image),
           "reference_interior_CV_percent": interior_cv_percent(reference_image),
           "baseline_interior_CV_percent": interior_cv_percent(baseline_image),
           "interior_pixels": int(interior.sum())}
first_simulation = {
    "Object": "Known uniform sphere slice; one finite box per pixel",
    "Voxel count": int(model.support.sum()),
    "Matrix / ADC samples": (model.image_size, model.nsamples),
    "Voxel size X/Y/Z [mm]": (*np.asarray(acquisition["FOV_m"])/np.array(model.image_size[::-1])*1000, SLICE_THICKNESS_MM),
    "TE / TR [ms]": (acquisition["TE_s"]*1000, acquisition["TR_s"]*1000),
    "ADC dwell [us]": acquisition["dwell_s"]*1e6,
    "RF flip [deg] / T1 / B1 / diffusion": "90 / infinity / 1 / 0",
    "PD at RF": PROTON_DENSITY,
    "T2* [ms]": "infinity" if np.isinf(T2STAR_MS) else T2STAR_MS,
    "Background B0 offset [Hz] / gradient [Hz/m]": (B0_OFFSET_HZ, B0_GRADIENT_HZ_PER_M),
    "Bubble radius / centre [mm]": (BUBBLE_RADIUS_MM, BUBBLE_CENTER_MM),
    "Main field [T] / direction / contrast [ppm]": (MAIN_FIELD_T, MAIN_FIELD_DIRECTION, BUBBLE_DELTA_CHI_PPM),
    "Total B0 range [Hz]": (np.nanmin(phantom["B0_Hz"]), np.nanmax(phantom["B0_Hz"])),
    "Bubble max corner field error [Hz]": phantom["bubble"]["corner_linearization_max_error_Hz"],
    "Bubble max corner phase error at TE [rad]": 2*np.pi*acquisition["TE_s"]*phantom["bubble"]["corner_linearization_max_error_Hz"],
    "Receive model": "One uniform receive channel",
    "Unresolved echo coherence": "None",
    "Device / precision": (str(model.device), str(model.dtype)),
    "Native MRzero version / precision": (model.runtime_info["mrzero_version"], "float32"),
    "Reconstruction": model.reconstruction.report["method"],
    "Setup + forward [s]": elapsed_s,
    "NRMSE versus numerical reference [%]": metrics["baseline_NRMSE_percent"],
}
table(first_simulation)
table(metrics)
with (OUTPUT_DIR/"mrzero_simulation.log").open("a") as log, redirect_stdout(log):
    quadrature = model.validate_voxel_quadrature(sample_count=8, point_count=8, quadrature_order=32)
print("Three-axis voxel integral check:", quadrature)
if not np.any(experiment["field"]["b0Hz"]) and not np.any(experiment["inplane_voxel"]["b0_gradient_Hz_per_pixel"]):
    assert metrics["baseline_NRMSE_percent"] < 1e-3, "Zero-B0 result differs unexpectedly from its reference."

In [ ]:
scale = float(np.abs(reference_image).max())
reference = np.abs(reference_image)/scale
baseline = np.abs(baseline_image)/scale
error = baseline-reference
maximum = max(float(reference.max()), float(baseline.max()))
error_limit = max(float(abs(error).max()), 1e-7)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), layout="constrained")
for ax, values, title in zip(axes[:2], [reference, baseline],
                            ["Zero-B0 reference", "Sphere, zero ACDC"]):
    im = image_ax(ax, values, title, cmap="gray", vmin=0, vmax=maximum)
    fig.colorbar(im, ax=ax, label="Magnitude / reference peak", shrink=.8)
im = image_ax(axes[2], error, f"Magnitude residual\nNRMSE {metrics['baseline_NRMSE_percent']:.4g}%",
              cmap="RdBu_r", vmin=-error_limit, vmax=error_limit)
fig.colorbar(im, ax=axes[2], label="Residual / reference peak", shrink=.8)
fig.savefig(OUTPUT_DIR/"sphere_simulation.png", dpi=300)
plt.show(); plt.close(fig)

## 5. Optional ACDC optimization

To test correction of the bubble perturbation, set `RUN_OPTIMIZATION = True`, then rerun from the parameter cell. This uses the existing projected image optimizer and the numerical zero-B0 target. Current, total-current and voxel-corner field limits are retained. The result is a simulation of a virtually placed sphere.

In [ ]:
currents_A = zero_currents.copy()
final_image, final_kdata = baseline_image, baseline_kdata
optimization = None
if RUN_OPTIMIZATION and metrics["baseline_NRMSE_percent"] > 1e-3:
    with (OUTPUT_DIR/"mrzero_simulation.log").open("a") as log, redirect_stdout(log):
        optimization = optimize_mrzero_image(
            model, zero_currents, total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
            field_limit_Hz=FIELD_LIMIT_HZ, max_iterations=MAX_ITERATIONS,
            learning_rate_A=.01, patience=MAX_ITERATIONS,
            use_cuda_graph=False, field_guidance_weight=0.)
    currents_A = optimization["currents_A"]
    final_image, final_kdata = optimization["image"], optimization["kdata"]
    metrics["optimized_NRMSE_percent"] = nrmse_percent(final_image, reference_image)
    print(f"Simulated NRMSE: {metrics['baseline_NRMSE_percent']:.4f}% -> {metrics['optimized_NRMSE_percent']:.4f}%")
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), layout="constrained")
    top = max(float(np.abs(baseline_image).max()/scale), float(np.abs(final_image).max()/scale))
    for ax, values, title in zip(axes, [baseline_image, final_image], ["Zero-current sphere", "Optimized sphere"]):
        im = image_ax(ax, np.abs(values)/scale, title, cmap="gray", vmin=0, vmax=top)
        fig.colorbar(im, ax=ax, label="Magnitude / reference peak", shrink=.8)
    fig.savefig(OUTPUT_DIR/"sphere_optimization.png", dpi=300)
    plt.show(); plt.close(fig)
else:
    print("Optimization disabled or the configured image already matches the reference.")
hardware_check = model.hardware.diagnostics(currents_A, TOTAL_CURRENT_LIMIT_A, FIELD_LIMIT_HZ)
assert hardware_check["all_limits_met"], "The waveform exceeds an enabled hardware limit."
print("All enabled current and field limits met:", hardware_check["all_limits_met"])

## 6. Save the known maps, complex data and first-simulation report

The NPZ retains both the exact input density and the finite-voxel numerical reference. The JSON records parameters, paths and metrics. No scanner waveform is exported.

In [ ]:
np.savez_compressed(
    OUTPUT_DIR/"sphere_results.npz", X_m=phantom["X_m"], Y_m=phantom["Y_m"],
    PD=phantom["PD"], B0_Hz=phantom["B0_Hz"], bubble_B0_Hz=phantom["bubble_B0_Hz"],
    B0_gradient_Hz_per_m=phantom["total_B0_gradient_Hz_per_m"],
    support=phantom["support"], interior_mask=interior,
    reference_image=reference_image, reference_kdata=model.ideal_kdata,
    baseline_image=baseline_image, baseline_kdata=baseline_kdata,
    final_image=final_image, final_kdata=final_kdata, currents_A=currents_A,
    kx_mInv=acquisition["kx_mInv"], ky_mInv=acquisition["ky_mInv"], time_s=acquisition["time_s"])
report = dict(
    generation="08 V3", phantom_geometry=phantom["geometry"], first_simulation=first_simulation,
    metrics=metrics, sequence_path=str(SEQUENCE_PATH), sequence_sha256=timing["sequence_sha256"],
    calibration_path=phantom["calibration_path"], calibration_sha256=phantom["calibration_sha256"],
    radius_mm=SPHERE_RADIUS_MM, slice_offset_mm=SLICE_OFFSET_MM, slice_thickness_mm=SLICE_THICKNESS_MM,
    coil_center_mm=COIL_CENTER_MM, bubble=phantom["bubble"],
    calibration_registration="virtual direct X/Y placement; translation only",
    calibration_excluded_voxels=phantom["excluded_calibration_voxels"],
    corner_linearization_max_error_Hz_per_A=phantom["corner_linearization_max_error_Hz_per_A"],
    quadrature=quadrature, optimization_performed=optimization is not None,
    all_enabled_hardware_limits_met=bool(hardware_check["all_limits_met"]))
def json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)
(OUTPUT_DIR/"sphere_report.json").write_text(json.dumps(report, indent=2, default=json_default, allow_nan=False))
print("Saved:", OUTPUT_DIR)